# 10. Сравнение моделей

Загружаем только сохранённые метрики, объединяем их и строим один понятный график. Повторного обучения нет.

## Подключение проекта

**Что делаем:** определяем корень проекта.  
**Зачем:** одинаковые пути должны работать локально и в Colab.  
**Что получим:** `PROJECT_ROOT` и доступный пакет из `src`.

In [1]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/fashion-recommender-system")
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(f"Не найдена папка src: {PROJECT_ROOT / 'src'}")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Корень проекта:", PROJECT_ROOT)

Корень проекта: <PROJECT_ROOT>


### Зависимости

**Что делаем:** устанавливаем requirements только в Colab.  
**Зачем:** локальное окружение не должно изменяться при каждом запуске.  
**Что получим:** готовые библиотеки для следующих ячеек.

In [2]:
import subprocess

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

### Импорты

**Что делаем:** подключаем pandas и matplotlib  
**Зачем:** notebook работает только с компактными CSV  
**Что получим:** два инструмента

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

### Пути к метрикам

**Что делаем:** перечисляем пять результатов notebooks 04–09  
**Зачем:** отсутствующая модель не заменяется нулём  
**Что получим:** `METRIC_FILES`

In [4]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "tables"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
TRANSACTIONS_PATH = RAW_DIR / "transactions_train.csv"
ARTICLES_PATH = RAW_DIR / "articles.csv"
CUSTOMERS_PATH = RAW_DIR / "customers.csv"
for directory in [PROCESSED_DIR, MODEL_DIR, REPORT_DIR, ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
METRIC_FILES = [
    REPORT_DIR / "baseline_metrics.csv",
    REPORT_DIR / "als_metrics.csv",
    REPORT_DIR / "content_metrics.csv",
    REPORT_DIR / "simple_hybrid_metrics.csv",
    REPORT_DIR / "catboost_metrics.csv",
]

### Проверка файлов

**Что делаем:** находим отсутствующие CSV  
**Зачем:** сравнение должно опираться только на реальные результаты  
**Что получим:** понятную ошибку с порядком notebooks

In [5]:
missing_metric_files = [
    path for path in METRIC_FILES if not path.is_file()
]
if missing_metric_files:
    raise FileNotFoundError(
        f"Не найдены metric files: {missing_metric_files}. "
        "Выполните notebooks 04–09 по порядку."
    )

### Загрузка метрик

**Что делаем:** читаем каждый CSV и объединяем строки  
**Зачем:** новые функции и модели здесь не нужны  
**Что получим:** `model_metrics`

In [6]:
metric_tables = [pd.read_csv(path) for path in METRIC_FILES]
model_metrics = pd.concat(
    metric_tables,
    ignore_index=True,
    sort=False,
)
print("Rows:", len(model_metrics))

Rows: 7


### Проверка моделей

**Что делаем:** смотрим названия и дубликаты  
**Зачем:** каждая model должна иметь одну строку, кроме трёх baseline в общем CSV  
**Что получим:** список фактических моделей

In [7]:
print(model_metrics["model"].tolist())
duplicate_models = model_metrics["model"].duplicated().sum()
print("Duplicate model names:", duplicate_models)
assert duplicate_models == 0

['Popularity', 'Recent Personal History', 'Frequent Personal History', 'ALS', 'Content-Based', 'Simple Hybrid', 'CatBoost Hybrid']
Duplicate model names: 0


### Таблица сравнения

**Что делаем:** выбираем одинаковые метрики и сортируем по Recall@12  
**Зачем:** NaN сохраняется как неприменимая метрика  
**Что получим:** понятную итоговую таблицу

In [8]:
comparison_columns = [
    "model", "Recall@12", "MAP@12", "HitRate@12",
    "Candidate Recall", "users_evaluated", "notes",
]
model_comparison = model_metrics.reindex(columns=comparison_columns)
model_comparison = model_comparison.sort_values(
    "Recall@12",
    ascending=False,
).reset_index(drop=True)
display(model_comparison)

                       model  ...                                   notes
0            CatBoost Hybrid  ...  CatBoostClassifier; common test cohort
1                 Popularity  ...          Top-12 baseline; common cohort
2    Recent Personal History  ...          Top-12 baseline; common cohort
3  Frequent Personal History  ...          Top-12 baseline; common cohort
4                        ALS  ...                  ALS Top-200 candidates
5              Simple Hybrid  ...           Fixed 0.5/0.3/0.1/0.1 weights
6              Content-Based  ...         Content-Based Top-50 candidates

[7 rows x 7 columns]


### График Top-12 метрик

**Что делаем:** строим Recall, MAP и HitRate одним grouped bar chart  
**Зачем:** позиционные и непозиционные метрики видны рядом  
**Что получим:** `model_comparison.png`

In [9]:
plot_table = model_comparison.set_index("model")[[
    "Recall@12", "MAP@12", "HitRate@12"
]]
axis = plot_table.plot.bar(figsize=(12, 5))
axis.set_title("Top-12 metrics on the common test cohort")
axis.set_ylabel("Metric value")
axis.set_xlabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
figure_path = FIGURE_DIR / "model_comparison.png"
plt.savefig(figure_path, dpi=150)
plt.show()
print("Сохранено:", figure_path)

Сохранено: <PROJECT_ROOT>/reports/figures/model_comparison.png


10_model_comparison_colab.ipynb:cell-18:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### Сохранение и вывод

**Что делаем:** записываем итоговую таблицу и называем лучший Recall  
**Зачем:** вывод основан на фактических значениях  
**Что получим:** `model_metrics.csv` и короткое заключение

In [10]:
comparison_path = REPORT_DIR / "model_metrics.csv"
model_comparison.to_csv(comparison_path, index=False)

best_model = model_comparison.iloc[0]
print("Лучший Recall@12:", best_model["model"])
print("Recall@12:", best_model["Recall@12"])
print("Сохранено:", comparison_path)

Лучший Recall@12: CatBoost Hybrid
Recall@12: 0.0171666666666666
Сохранено: <PROJECT_ROOT>/reports/tables/model_metrics.csv
